# acoustic proximity sensor

In [13]:
# Unzip file
# !unzip "/content/dataset.zip" -d "/content/dataset"


In [14]:
# install ffmpeg for mp3 audio
# !pip install torchaudio librosa
# !apt-get install -y ffmpeg

## Run Model

In [15]:
# Run this cell first to install required MP3 decoders and libraries
!apt-get update -qq && !apt-get install -y ffmpeg
!pip install torchaudio librosa torchvision

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
/bin/bash: line 1: !apt-get: command not found


Imports & Helper Functions

In [16]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models

# Configure torchaudio to use ffmpeg backend for MP3 files
try:
    torchaudio.set_audio_backend("ffmpeg")
except Exception:
    pass

def extract_spectrogram(file_path, target_sample_rate=16000, duration_seconds=1.0):
    """
    Loads an MP3 audio file, normalizes duration/channels, and converts to Mel-Spectrogram.
    """
    try:
        waveform, sample_rate = torchaudio.load(file_path, format="mp3")
    except Exception:
        # Fallback to librosa if torchaudio encounters decoding issues with specific MP3 formats
        import librosa
        y, sample_rate = librosa.load(file_path, sr=target_sample_rate)
        waveform = torch.tensor(y).unsqueeze(0)

    # Resample if sample rate doesn't match target
    if sample_rate != target_sample_rate:
        resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
        waveform = resampler(waveform)

    # Convert stereo to mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # Pad or trim to ensure uniform sequence length
    target_samples = int(target_sample_rate * duration_seconds)
    current_samples = waveform.shape[1]

    if current_samples < target_samples:
        padding = target_samples - current_samples
        waveform = torch.nn.functional.pad(waveform, (0, padding))
    elif current_samples > target_samples:
        waveform = waveform[:, :target_samples]

    # Convert to Mel Spectrogram
    mel_spectrogram = T.MelSpectrogram(
        sample_rate=target_sample_rate,
        n_fft=1024,
        hop_length=512,
        n_mels=64
    )(waveform)

    # Convert to Decibel (dB) scale
    log_mel = T.AmplitudeToDB()(mel_spectrogram)
    return log_mel


class AcousticDataset(Dataset):
    """
    Dataset class to load MP3 files from subdirectories.
    """
    def __init__(self, data_dir):
        self.file_paths = []
        self.labels = []
        self.classes = ['no_object', 'object_present']

        for label, category in enumerate(self.classes):
            category_dir = os.path.join(data_dir, category)
            if not os.path.exists(category_dir):
                continue
            for file_name in os.listdir(category_dir):
                if file_name.lower().endswith('.mp3'):
                    self.file_paths.append(os.path.join(category_dir, file_name))
                    self.labels.append(label)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        spectrogram = extract_spectrogram(path)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return spectrogram, label


class AcousticClassifier(nn.Module):
    """
    ResNet-18 model modified to accept 1-channel spectrograms.
    """
    def __init__(self):
        super(AcousticClassifier, self).__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Modify first layer for 1-channel input
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # Binary classification output layer
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, 2)

    def forward(self, x):
        return self.backbone(x)

print("Helper classes and functions loaded successfully.")

Helper classes and functions loaded successfully.


Directory Structure Setup

In [17]:
# Create directory structure inside Colab
import os

base_dir = "/content/dataset"
train_no_obj = os.path.join(base_dir, "train/no_object")
train_obj = os.path.join(base_dir, "train/object_present")

os.makedirs(train_no_obj, exist_ok=True)
os.makedirs(train_obj, exist_ok=True)

print("Directory structure created successfully:")
print(f"1. Put MP3 files WITHOUT objects here: {train_no_obj}")
print(f"2. Put MP3 files WITH objects here:    {train_obj}")

Directory structure created successfully:
1. Put MP3 files WITHOUT objects here: /content/dataset/train/no_object
2. Put MP3 files WITH objects here:    /content/dataset/train/object_present


Model Training

In [20]:
def train_model(data_dir, save_path="/content/acoustic_model.pth", epochs=100, batch_size=8, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on device: {device}")

    dataset = AcousticDataset(data_dir)
    if len(dataset) == 0:
        raise ValueError(f"No MP3 files found in '{data_dir}'. Upload files to the subdirectories and try again.")

    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = AcousticClassifier().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    print("--- Starting Training ---")
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / total
        epoch_acc = (correct / total) * 100
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.2f}%")

    torch.save(model.state_dict(), save_path)
    print(f"\nModel training complete. Weights saved to: {save_path}")

# Run training
DATASET_DIR = "/content/dataset/train"
MODEL_SAVE_PATH = "/content/acoustic_model.pth"

train_model(data_dir=DATASET_DIR, save_path=MODEL_SAVE_PATH, epochs=100)

Training on device: cuda
--- Starting Training ---
Epoch [1/100] - Loss: 0.6186 - Accuracy: 58.33%
Epoch [2/100] - Loss: 0.4833 - Accuracy: 91.67%
Epoch [3/100] - Loss: 0.4078 - Accuracy: 66.67%
Epoch [4/100] - Loss: 0.0946 - Accuracy: 91.67%
Epoch [5/100] - Loss: 0.0053 - Accuracy: 100.00%
Epoch [6/100] - Loss: 0.3860 - Accuracy: 91.67%
Epoch [7/100] - Loss: 0.6013 - Accuracy: 83.33%
Epoch [8/100] - Loss: 0.0069 - Accuracy: 100.00%
Epoch [9/100] - Loss: 1.7061 - Accuracy: 75.00%
Epoch [10/100] - Loss: 0.5120 - Accuracy: 83.33%
Epoch [11/100] - Loss: 0.0032 - Accuracy: 100.00%
Epoch [12/100] - Loss: 0.0131 - Accuracy: 100.00%
Epoch [13/100] - Loss: 0.2210 - Accuracy: 91.67%
Epoch [14/100] - Loss: 0.0332 - Accuracy: 100.00%
Epoch [15/100] - Loss: 0.8358 - Accuracy: 83.33%
Epoch [16/100] - Loss: 0.2860 - Accuracy: 91.67%
Epoch [17/100] - Loss: 0.0496 - Accuracy: 100.00%
Epoch [18/100] - Loss: 0.1010 - Accuracy: 91.67%
Epoch [19/100] - Loss: 0.0186 - Accuracy: 100.00%
Epoch [20/100] - Los

Run Prediction on New MP3 File

In [21]:
def predict(audio_file_path, model_path="/content/acoustic_model.pth"):
    if not os.path.exists(audio_file_path):
        print(f"Error: Target MP3 file not found at '{audio_file_path}'")
        return
    if not os.path.exists(model_path):
        print(f"Error: Trained model weights not found at '{model_path}'")
        return

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    class_names = ['No Object', 'Object Present']

    model = AcousticClassifier().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    spectrogram = extract_spectrogram(audio_file_path)
    input_tensor = spectrogram.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_class = torch.max(probabilities, dim=1)

    result_label = class_names[predicted_class.item()]
    confidence_score = confidence.item() * 100

    print(f"\n--- Prediction Results ---")
    print(f"File Path:  {audio_file_path}")
    print(f"Detected:   {result_label}")
    print(f"Confidence: {confidence_score:.2f}%")

# Specify the path to your new test MP3 file and execute prediction
TEST_FILE = "/content/test_sample.mp3"
predict(audio_file_path=TEST_FILE, model_path=MODEL_SAVE_PATH)


--- Prediction Results ---
File Path:  /content/test_sample.mp3
Detected:   Object Present
Confidence: 98.54%
